## **Lab Exercise #15**
- Name: Michael Kenny
- Date Started: 10/12/2025
- Date Completed: 10/12/2025
- Class: CSC-180-02
- Professor: Dr. Mamoun Abu-Samaha

**Assignment Objective:**
- Download the heart disease dataset from Kaggle.
- Remove outliers using Z-score.
    - Remove anything that has a Z score > 3 or Z score < -3.
- Convert text columns to numbers using label encoding and OHE.
- Apply dimensionality scaling.
- Build a classification model using various models (SVM, logistical regression, random forest) to see which
   gives the best accuracy.
- Use PCA to reduce dimensions, retrain models and see what kind of impact the reduction had.

**Test Cases/Predictions:**
- Calculate the score of the model with (SVM, LR, and RandomForest) using both the PCA reduced form and original forms.

### **1. Read the CSV File into Dataframe**

In [562]:
import pandas as pd

df = pd.read_csv('./heart.csv')

### **2. Z-Score (Finding Outliers)**

In [563]:
df['zAge'] = (df['Age'] - df['Age'].mean()) / df['Age'].std()
df['zRestingBP'] = (df['RestingBP'] - df['RestingBP'].mean()) / df['RestingBP'].std()
df['zCholesterol'] = (df['Cholesterol'] - df['Cholesterol'].mean()) / df['Cholesterol'].std()
df['zMaxHR'] = (df['MaxHR'] - df['MaxHR'].mean()) / df['MaxHR'].std()
df

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,zAge,zRestingBP,zCholesterol,zMaxHR
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0,-1.432359,0.410685,0.824621,1.382175
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1,-0.478223,1.490940,-0.171867,0.753746
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0,-1.750404,-0.129442,0.769768,-1.524307
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1,-0.584238,0.302660,0.138964,-1.131539
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0,0.051853,0.950812,-0.034736,-0.581664
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1,-0.902283,-1.209697,0.596068,-0.188897
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1,1.536064,0.626736,-0.053020,0.164595
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1,0.369898,-0.129442,-0.619830,-0.856602
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1,0.369898,-0.129442,0.340090,1.460728


This section above refers to the Z-score formula for finding the statistical score based on the mean and standard deviations for each column that is significant. We exclude z-scores in the next step that are too great or too small, which indicates they are far away from the center of the distribution.

In [564]:
zCategories = ('zAge', 'zRestingBP', 'zCholesterol', 'zMaxHR')
for category in zCategories:
    df = df[df[category] < 3]
    df = df[df[category] > -3]
df

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,zAge,zRestingBP,zCholesterol,zMaxHR
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0,-1.432359,0.410685,0.824621,1.382175
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1,-0.478223,1.490940,-0.171867,0.753746
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0,-1.750404,-0.129442,0.769768,-1.524307
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1,-0.584238,0.302660,0.138964,-1.131539
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0,0.051853,0.950812,-0.034736,-0.581664
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1,-0.902283,-1.209697,0.596068,-0.188897
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1,1.536064,0.626736,-0.053020,0.164595
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1,0.369898,-0.129442,-0.619830,-0.856602
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1,0.369898,-0.129442,0.340090,1.460728


New dataframe with outlier rows removed (12 rows were removed).

In [565]:
for category in zCategories:
    df = df.drop([category], axis='columns')
df

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1


Drop the z-score tables now that outliers are removed.

### **3. Label Encoding Ordinal Columns**

In [566]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
ecgLE = le.fit_transform(df['RestingECG'])
exerciseLE = le.fit_transform(df['ExerciseAngina'])
slopeLE = le.fit_transform(df['ST_Slope'])
df['RestingECG'] = ecgLE
df['ExerciseAngina'] = exerciseLE
df['ST_Slope'] = slopeLE
df

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,1,172,0,0.0,2,0
1,49,F,NAP,160,180,0,1,156,0,1.0,1,1
2,37,M,ATA,130,283,0,2,98,0,0.0,2,0
3,48,F,ASY,138,214,0,1,108,1,1.5,1,1
4,54,M,NAP,150,195,0,1,122,0,0.0,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,1,132,0,1.2,1,1
914,68,M,ASY,144,193,1,1,141,0,3.4,1,1
915,57,M,ASY,130,131,0,1,115,1,1.2,1,1
916,57,F,ATA,130,236,0,0,174,0,0.0,1,1


The RestingECG, ExerciseAngina, and ST_Slope columns are all ordinal relative to each other. We can use label encoding here and move on.

In [567]:
dummies_chest = pd.get_dummies(df['Sex'])
dummies_ECG = pd.get_dummies(df['ChestPainType'])
df = df.drop(['Sex', 'ChestPainType'], axis='columns')
df = pd.concat([df, dummies_chest, dummies_ECG], axis='columns')
df

,Age,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,F,M,ASY,ATA,NAP,TA
0,40,140,289,0,1,172,0,0.0,2,0,False,True,False,True,False,False
1,49,160,180,0,1,156,0,1.0,1,1,True,False,False,False,True,False
2,37,130,283,0,2,98,0,0.0,2,0,False,True,False,True,False,False
3,48,138,214,0,1,108,1,1.5,1,1,True,False,True,False,False,False
4,54,150,195,0,1,122,0,0.0,2,0,False,True,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,110,264,0,1,132,0,1.2,1,1,False,True,False,False,False,True
914,68,144,193,1,1,141,0,3.4,1,1,False,True,True,False,False,False
915,57,130,131,0,1,115,1,1.2,1,1,False,True,True,False,False,False
916,57,130,236,0,0,174,0,0.0,1,1,True,False,False,True,False,False


The Sex and ChestPainType columns are nominal string values, relative to each other they are meaningless other than to classify one different or similar to the other. We can use dummies or OHE here.

### **4. Setup Variables**

In [568]:
X = df.drop(['HeartDisease'], axis='columns')
y = df.HeartDisease

#### **Scale Data**

In [569]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled

array([[-1.43268734,  0.46074981,  0.848846  , ...,  2.07322107,
        -0.5322463 , -0.23127553],
       [-0.47787471,  1.62434375, -0.16934854, ..., -0.48234123,
         1.87882942, -0.23127553],
       [-1.75095822, -0.12104716,  0.79279859, ...,  2.07322107,
        -0.5322463 , -0.23127553],
       ...,
       [ 0.37084763, -0.12104716, -0.62706901, ..., -0.48234123,
        -0.5322463 , -0.23127553],
       [ 0.37084763, -0.12104716,  0.35376058, ...,  2.07322107,
        -0.5322463 , -0.23127553],
       [-1.64486793,  0.34439041, -0.21605471, ..., -0.48234123,
         1.87882942, -0.23127553]], shape=(906, 15))

### **5. Split Data**

In [570]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)

### **6. Train/Score Models on Scaled Data**

#### **Logistical Regression**

In [571]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(C=14, max_iter=2000)
model.fit(X_train, y_train)
model.score(X_test, y_test)

0.8461538461538461

#### **Support Vector Machine**

In [572]:
from sklearn.svm import SVC

model = SVC(kernel='rbf', C=40, gamma=0.2)
model.fit(X_train, y_train)
model.score(X_test, y_test)

0.7747252747252747

#### **Random Forest**

In [573]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)
model.score(X_test, y_test)

0.8461538461538461

### **7. Use PCA (Principle Component Analysis) To Dimensionalize Data**

In [574]:
from sklearn.decomposition import PCA

pca = PCA(0.95)
X_pca = pca.fit_transform(X_scaled)

### **8. Re-Split Data**

In [575]:
X_train_pca, X_test_pca, y_train, y_test = train_test_split(X_pca, y, test_size=0.2)

### **9. Train/Score Models on PCA Scaled Data**

#### **Logistical Regression**

In [576]:
model = LogisticRegression(C=45, max_iter=2000)
model.fit(X_train_pca, y_train)
model.score(X_test_pca, y_test)

0.8681318681318682

#### **Support Vector Machine**

In [577]:
model = SVC(kernel='rbf', C=30)
model.fit(X_train_pca, y_train)
model.score(X_test_pca, y_test)

0.9065934065934066

#### **Random Forest**

In [578]:
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train_pca, y_train)
model.score(X_test_pca, y_test)

0.8681318681318682

### **10. Conclusion**

This lab exercise was fun since I got to use many different models with the same data and tune them here and there to see how they compare. Comparing the models with the scalar scaled data and the PCA data was also interesting to see, the scores really didn't differ a whole bunch except for the SVM model which improved. Overall I am hoping I didn't drop the wrong columns and used the label encoding and dummy variables properly on the correct ordinal/nominal value sets.